# silver_commission_payouts

The showcase notebook. This is the PySpark replacement for the legacy VBA tool that computed agent commissions for the past 15 years. The VBA tool had no documentation, no validation, and lived in one person's head. This notebook documents every rule explicitly, pulls from source-of-truth silver tables instead of trusting recorded values, and produces a recomputed commission amount alongside the original VBA figure so drift is visible.

The output of this notebook feeds directly into silver_commission_reconciliation, which surfaces the ~3% of rows where VBA diverged from what it should have calculated.

## The five documented business rules

Every commission calculation applies these rules in order:

1. **Tier rate** — agent's commission rate by tier. Looked up from silver_agents, not trusted from bronze. BRONZE 0.08, SILVER 0.10, GOLD 0.12, PLATINUM 0.15.

2. **Program modifier** — per-program multiplier applied to gross commission. Looked up from silver_programs. HCP 1.15, CONT 1.20, PROP 0.95, BIZ 0.90, others between.

3. **Volume bonus** — agents whose trailing 12-month written premium exceeds $850K get +0.01 added to their tier rate. Recomputed from silver_policies, not trusted from bronze.

4. **Status override** — CANCELLED policies get a 0.0 multiplier (full clawback). ACTIVE and EXPIRED get 1.0.

5. **Ceiling** — commission_amount is capped at $50,000 regardless of premium size.

Formula:
    effective_rate     = tier_rate + volume_bonus
    gross              = premium x effective_rate x program_modifier x status_multiplier
    gross_capped       = min(gross, 50000)
    silver_commission  = gross_capped - adjustments

## What's different from bronze

Bronze records what the VBA tool used — including drift. This notebook recomputes from silver source-of-truth tables. Both values live on the same row so silver_commission_reconciliation can compare them directly.

## Sources

- `bronze_citadel_mga.bronze_commission_payouts` (5,000 rows, VBA-recorded values)
- `silver_citadel_mga.silver_agents` (true tier rates)
- `silver_citadel_mga.silver_programs` (true program modifiers)
- `silver_citadel_mga.silver_policies` (trailing premium for volume bonus)

## Output

- `silver_citadel_mga.silver_commission_payouts` (5,000 rows with both bronze and silver commission values)


In [1]:
from pyspark.sql.functions import (
    lit, current_timestamp, col, when, sum as spark_sum,
    greatest, least
)
from pyspark.sql.types import DoubleType
import uuid
from datetime import datetime, timedelta

SILVER_BATCH_ID = f"silver_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

# Source: bronze commission payouts (what VBA recorded).
SOURCE_BRONZE = "bronze_citadel_mga.dbo.bronze_commission_payouts"

# Silver reference tables for source-of-truth lookups.
REF_AGENTS    = "silver_citadel_mga.dbo.silver_agents"
REF_PROGRAMS  = "silver_citadel_mga.dbo.silver_programs"
REF_POLICIES  = "silver_citadel_mga.dbo.silver_policies"

# Output table.
TARGET_TABLE  = "silver_commission_payouts"

# Commission rule constants — same values as the Phase 1 generator.
# Documenting them here is the point: these are the rules, explicit and versioned.
VOLUME_BONUS_THRESHOLD = 850_000.0
VOLUME_BONUS_AMOUNT    = 0.01
COMMISSION_CEILING     = 50_000.0

# Trailing window for volume bonus: 12 months back from the end of the data range.
TRAILING_WINDOW_START  = "2025-01-01"

# Status multipliers.
STATUS_MULTIPLIERS = {
    "ACTIVE":    1.0,
    "EXPIRED":   1.0,
    "CANCELLED": 0.0,
}

print(f"Silver batch ID:          {SILVER_BATCH_ID}")
print(f"Source:                   {SOURCE_BRONZE}")
print(f"Target:                   {TARGET_TABLE}")
print(f"Volume bonus threshold:   ${VOLUME_BONUS_THRESHOLD:,.0f}")
print(f"Volume bonus amount:      +{VOLUME_BONUS_AMOUNT}")
print(f"Commission ceiling:       ${COMMISSION_CEILING:,.0f}")
print(f"Trailing window start:    {TRAILING_WINDOW_START}")

StatementMeta(, 62bc6e45-f636-40fd-8de1-22236b6bccf7, 3, Finished, Available, Finished, False)

Silver batch ID:          silver_20260603_022915_bc81b40d
Source:                   bronze_citadel_mga.dbo.bronze_commission_payouts
Target:                   silver_commission_payouts
Volume bonus threshold:   $850,000
Volume bonus amount:      +0.01
Commission ceiling:       $50,000
Trailing window start:    2025-01-01


In [2]:
# Read bronze commission payouts — what VBA recorded.
bronze_df = spark.read.table(SOURCE_BRONZE)

print(f"Bronze commission payouts: {bronze_df.count()} rows, {len(bronze_df.columns)} cols")
print(f"Columns: {bronze_df.columns}")
print()

# Load silver reference tables for source-of-truth lookups.
agents_df   = spark.read.table(REF_AGENTS)
programs_df = spark.read.table(REF_PROGRAMS)
policies_df = spark.read.table(REF_POLICIES)

print(f"silver_agents loaded:    {agents_df.count()} rows")
print(f"silver_programs loaded:  {programs_df.count()} rows")
print(f"silver_policies loaded:  {policies_df.count()} rows")
print()

# Build Python dicts for agent and program lookups.
# Collecting small reference tables to driver is fine here.
# Don't do this for large tables.
agent_true_rate = {
    row["agent_id"]: row["commission_rate"]
    for row in agents_df.select("agent_id", "commission_rate").collect()
}

program_true_modifier = {
    row["program_code"]: row["commission_modifier"]
    for row in programs_df.select("program_code", "commission_modifier").collect()
}

print(f"Agent rate lookup built:    {len(agent_true_rate)} agents")
print(f"Program modifier lookup built: {len(program_true_modifier)} programs")
print()
print(f"Sample agent rates:    {dict(list(agent_true_rate.items())[:3])}")
print(f"Sample program mods:   {program_true_modifier}")

StatementMeta(, 62bc6e45-f636-40fd-8de1-22236b6bccf7, 4, Finished, Available, Finished, False)

Bronze commission payouts: 5000 rows, 19 cols
Columns: ['payout_id', 'policy_id', 'agent_id', 'program_code', 'premium', 'commission_rate', 'program_modifier', 'volume_bonus', 'policy_status', 'status_multiplier', 'adjustments', 'commission_amount', 'payout_date', 'payout_status', 'calc_method', '_ingestion_timestamp', '_source_file', '_batch_id', '_ingestion_method']

silver_agents loaded:    200 rows
silver_programs loaded:  10 rows
silver_policies loaded:  9709 rows

Agent rate lookup built:    200 agents
Program modifier lookup built: 10 programs

Sample agent rates:    {'AGT-001': 0.08, 'AGT-002': 0.08, 'AGT-003': 0.08}
Sample program mods:   {'DIET': 1.05, 'ENV': 1.05, 'HCP': 1.15, 'ENRG': 1.1, 'LIFE': 1.1, 'CONT': 1.2, 'MFG': 1.0, 'PROP': 0.95, 'UMB': 1.0, 'BIZ': 0.9}


In [3]:
# Recompute volume bonus qualifiers from silver_policies.
# An agent qualifies if their trailing 12-month written premium > $850K.
# This is Rule 3 of the five documented commission rules.
#
# We compute this from silver_policies, not from the bronze recorded
# volume_bonus column. That's the point — Pattern C drift exists because
# VBA applied the bonus to agents who didn't qualify. Silver finds the truth.

trailing_policies = policies_df.filter(
    col("effective_date") >= TRAILING_WINDOW_START
)

agent_trailing_premium = (
    trailing_policies
    .groupBy("agent_id")
    .agg(spark_sum("premium").alias("trailing_12mo_premium"))
)

qualifying_agents_df = agent_trailing_premium.filter(
    col("trailing_12mo_premium") > VOLUME_BONUS_THRESHOLD
)

# Collect qualifying agent IDs into a Python set for row-level lookup.
qualifying_agent_ids = set(
    row["agent_id"] for row in qualifying_agents_df.collect()
)

print(f"Policies in trailing 12-month window: {trailing_policies.count()}")
print(f"Agents with any trailing premium:     {agent_trailing_premium.count()}")
print(f"Qualifying agents (> $850K):          {len(qualifying_agent_ids)}")
print()
print("Top 5 agents by trailing premium:")
agent_trailing_premium.orderBy("trailing_12mo_premium", ascending=False).show(5)

StatementMeta(, 62bc6e45-f636-40fd-8de1-22236b6bccf7, 5, Finished, Available, Finished, False)

Policies in trailing 12-month window: 4896
Agents with any trailing premium:     200
Qualifying agents (> $850K):          16

Top 5 agents by trailing premium:
+--------+---------------------+
|agent_id|trailing_12mo_premium|
+--------+---------------------+
| AGT-077|           1143803.25|
| AGT-098|            991206.28|
| AGT-143|    981755.2599999999|
| AGT-147|            964938.87|
| AGT-185|    947633.4600000002|
+--------+---------------------+
only showing top 5 rows



In [5]:
# Recompute commission amount for every payout row using source-of-truth
# inputs from silver reference tables. Bronze recorded values (what VBA used)
# are kept alongside so silver_commission_reconciliation can compare them.

def get_true_rate(agent_id):
    return agent_true_rate.get(agent_id, None)

def get_true_modifier(program_code):
    return program_true_modifier.get(program_code, None)

def get_volume_bonus(agent_id):
    return VOLUME_BONUS_AMOUNT if agent_id in qualifying_agent_ids else 0.0

def get_status_multiplier(status):
    return STATUS_MULTIPLIERS.get(status, 1.0)

# Register as Spark UDFs so they can be applied to DataFrame columns.
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

udf_true_rate       = udf(get_true_rate, DoubleType())
udf_true_modifier   = udf(get_true_modifier, DoubleType())
udf_volume_bonus    = udf(get_volume_bonus, DoubleType())
udf_status_mult     = udf(get_status_multiplier, DoubleType())

# Apply UDFs to get source-of-truth values for each row.
with_true_inputs = (
    bronze_df
    .withColumn("true_tier_rate",       udf_true_rate(col("agent_id")))
    .withColumn("true_program_modifier", udf_true_modifier(col("program_code")))
    .withColumn("true_volume_bonus",    udf_volume_bonus(col("agent_id")))
    .withColumn("true_status_multiplier", udf_status_mult(col("policy_status")))
)

# Apply the five rules using true inputs.
with_silver_commission = (
    with_true_inputs
    # Rule 1 + 3: effective rate = tier rate + volume bonus
    .withColumn(
        "silver_effective_rate",
        col("true_tier_rate") + col("true_volume_bonus")
    )
    # Rules 2 + 4: gross = premium x effective_rate x program_modifier x status_multiplier
    .withColumn(
        "silver_gross",
        col("premium") * col("silver_effective_rate") *
        col("true_program_modifier") * col("true_status_multiplier")
    )
    # Rule 5: ceiling at $50,000
    .withColumn(
        "silver_gross_capped",
        when(col("silver_gross") > COMMISSION_CEILING, COMMISSION_CEILING)
        .otherwise(col("silver_gross"))
    )
    # Final: subtract adjustments (same as bronze — adjustments are trusted)
    .withColumn(
        "silver_commission_amount",
        (col("silver_gross_capped") - col("adjustments")).cast("decimal(15,2)").cast("double")
    )
    # Drop intermediate calculation columns — keep inputs and outputs only.
    .drop("silver_gross", "silver_gross_capped")
)

print(f"Rows processed: {with_silver_commission.count()}")
print()
print("Sample — comparing bronze vs silver commission:")
with_silver_commission.select(
    "payout_id",
    "agent_id",
    "program_code",
    "commission_rate",        # what VBA used
    "true_tier_rate",         # source of truth
    "program_modifier",       # what VBA used
    "true_program_modifier",  # source of truth
    "commission_amount",      # what VBA calculated
    "silver_commission_amount" # what silver calculates
).show(10, truncate=False)

StatementMeta(, 62bc6e45-f636-40fd-8de1-22236b6bccf7, 7, Finished, Available, Finished, False)

Rows processed: 5000

Sample — comparing bronze vs silver commission:
+-----------+--------+------------+---------------+--------------+----------------+---------------------+-----------------+------------------------+
|payout_id  |agent_id|program_code|commission_rate|true_tier_rate|program_modifier|true_program_modifier|commission_amount|silver_commission_amount|
+-----------+--------+------------+---------------+--------------+----------------+---------------------+-----------------+------------------------+
|CMP-0000001|AGT-150 |LIFE        |0.08           |0.08          |1.1             |1.1                  |1458.47          |1458.47                 |
|CMP-0000002|AGT-149 |ENV         |0.1            |0.1           |1.05            |1.05                 |1649.09          |1649.09                 |
|CMP-0000003|AGT-128 |ENRG        |0.08           |0.08          |1.1             |1.1                  |2640.0           |2640.0                  |
|CMP-0000004|AGT-076 |ENV         |0

In [6]:
# Drop bronze lineage columns and add silver lineage before writing.
silver_df = (
    with_silver_commission
    .withColumnRenamed("_batch_id", "_bronze_batch_id")
    .drop("_ingestion_timestamp", "_source_file", "_ingestion_method")
    .withColumn("_silver_load_timestamp", current_timestamp())
    .withColumn("_silver_batch_id", lit(SILVER_BATCH_ID))
)

(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE)
)

print(f"Wrote {silver_df.count()} rows to {TARGET_TABLE}")
print(f"Columns: {silver_df.columns}")

StatementMeta(, 62bc6e45-f636-40fd-8de1-22236b6bccf7, 8, Finished, Available, Finished, False)

Wrote 5000 rows to silver_commission_payouts
Columns: ['payout_id', 'policy_id', 'agent_id', 'program_code', 'premium', 'commission_rate', 'program_modifier', 'volume_bonus', 'policy_status', 'status_multiplier', 'adjustments', 'commission_amount', 'payout_date', 'payout_status', 'calc_method', '_bronze_batch_id', 'true_tier_rate', 'true_program_modifier', 'true_volume_bonus', 'true_status_multiplier', 'silver_effective_rate', 'silver_commission_amount', '_silver_load_timestamp', '_silver_batch_id']


In [7]:
verify_df = spark.read.table(TARGET_TABLE)

row_count = verify_df.count()
col_count = len(verify_df.columns)

required_cols = [
    "commission_amount",       # what VBA recorded
    "silver_commission_amount", # what silver recomputed
    "true_tier_rate",
    "true_program_modifier",
    "true_volume_bonus",
    "_bronze_batch_id",
    "_silver_batch_id"
]
has_required = all(c in verify_df.columns for c in required_cols)

print(f"silver_commission_payouts verified:")
print(f"  Rows:            {row_count}")
print(f"  Columns:         {col_count}")
print(f"  Required cols:   {'present' if has_required else 'MISSING'}")
print()

# Quick drift preview — rows where bronze and silver amounts differ.
drift_df = verify_df.filter(
    col("commission_amount") != col("silver_commission_amount")
)
drift_count = drift_df.count()
drift_pct   = drift_count / row_count * 100

print(f"Rows with drift:   {drift_count} ({drift_pct:.1f}%)")
print()
print("Sample drift rows:")
drift_df.select(
    "payout_id", "agent_id", "program_code",
    "commission_rate", "true_tier_rate",
    "program_modifier", "true_program_modifier",
    "volume_bonus", "true_volume_bonus",
    "commission_amount", "silver_commission_amount"
).show(5, truncate=False)

StatementMeta(, 62bc6e45-f636-40fd-8de1-22236b6bccf7, 9, Finished, Available, Finished, False)

silver_commission_payouts verified:
  Rows:            5000
  Columns:         24
  Required cols:   present

Rows with drift:   110 (2.2%)

Sample drift rows:
+-----------+--------+------------+---------------+--------------+----------------+---------------------+------------+-----------------+-----------------+------------------------+
|payout_id  |agent_id|program_code|commission_rate|true_tier_rate|program_modifier|true_program_modifier|volume_bonus|true_volume_bonus|commission_amount|silver_commission_amount|
+-----------+--------+------------+---------------+--------------+----------------+---------------------+------------+-----------------+-----------------+------------------------+
|CMP-0000007|AGT-049 |PROP        |0.12           |0.12          |0.95            |0.95                 |0.01        |0.0              |12350.0          |11400.0                 |
|CMP-0000041|AGT-123 |UMB         |0.1            |0.1           |1.0             |1.0                  |0.01        |0.